# HireSense AI - Resume NER Training

Train a BERT + BiLSTM + CRF model for Named Entity Recognition on resumes.

**Model Architecture:**
- BERT-base-uncased (768-dim contextual embeddings)
- BiLSTM (2 layers, 256 hidden units, bidirectional)
- CRF layer for sequence labeling

**Entity Types:**
- SKILL, EXP, EDU, PROJ, ACH, ORG, LOC, DATE, NAME, CONTACT, CERT

## 1. Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch transformers pytorch-crf seqeval datasets pandas scikit-learn tqdm pdfplumber

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive (for saving models)
from google.colab import drive
drive.mount('/content/drive')

## 2. Download Datasets

In [ ]:
# Create data directory
!mkdir -p data/resume_corpus data/ner_annotated_cvs data/kaggle_resume_pdf

In [ ]:
# Download Resume Corpus dataset
# Option 1: From GitHub
!git clone https://github.com/vrundag91/Resume-Corpus-Dataset.git data/resume_corpus_raw || echo "Already exists"

# Option 2: From NER-Annotated-CVs
!git clone https://github.com/mehyarmlaweh/ner-annotated-cvs.git data/ner_annotated_raw || echo "Already exists"

In [ ]:
# For Kaggle dataset, you need to:
# 1. Go to https://www.kaggle.com/datasets/hadikp/resume-data-pdf
# 2. Download and upload to Colab, or use Kaggle API:

# Uncomment and run if you have Kaggle credentials:
# !pip install kaggle
# !mkdir -p ~/.kaggle
# # Upload your kaggle.json file first
# !kaggle datasets download -d hadikp/resume-data-pdf -p data/kaggle_resume_pdf --unzip

## 3. Configuration

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class ModelConfig:
    bert_model_name: str = "bert-base-uncased"
    bert_hidden_size: int = 768
    freeze_bert: bool = False
    lstm_hidden_size: int = 256
    lstm_num_layers: int = 2
    lstm_dropout: float = 0.3
    lstm_bidirectional: bool = True
    
    @property
    def lstm_output_size(self) -> int:
        return self.lstm_hidden_size * (2 if self.lstm_bidirectional else 1)

@dataclass
class TrainingConfig:
    max_seq_length: int = 512
    train_batch_size: int = 16
    eval_batch_size: int = 32
    bert_learning_rate: float = 2e-5
    lstm_crf_learning_rate: float = 1e-3
    weight_decay: float = 0.01
    adam_epsilon: float = 1e-8
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.1
    num_epochs: int = 10
    early_stopping_patience: int = 3
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    seed: int = 42

# Entity labels
ENTITY_LABELS = [
    "O", "B-SKILL", "I-SKILL", "B-EXP", "I-EXP", "B-EDU", "I-EDU",
    "B-PROJ", "I-PROJ", "B-ACH", "I-ACH", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-DATE", "I-DATE", "B-NAME", "I-NAME",
    "B-CONTACT", "I-CONTACT", "B-CERT", "I-CERT"
]

LABEL2ID = {label: idx for idx, label in enumerate(ENTITY_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
NUM_LABELS = len(ENTITY_LABELS)

model_config = ModelConfig()
train_config = TrainingConfig()

print(f"Device: {train_config.device}")
print(f"Num labels: {NUM_LABELS}")

## 4. Model Architecture

In [ ]:
import torch.nn as nn
from transformers import BertModel
from torchcrf import CRF

class BertBiLSTMCRF(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.num_labels = NUM_LABELS
        
        # BERT
        self.bert = BertModel.from_pretrained(config.bert_model_name)
        if config.freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False
        
        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=config.bert_hidden_size,
            hidden_size=config.lstm_hidden_size,
            num_layers=config.lstm_num_layers,
            batch_first=True,
            bidirectional=config.lstm_bidirectional,
            dropout=config.lstm_dropout if config.lstm_num_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(config.lstm_dropout)
        self.hidden2tag = nn.Linear(config.lstm_output_size, self.num_labels)
        self.crf = CRF(self.num_labels, batch_first=True)
        
    def forward(self, input_ids, attention_mask, labels=None):
        bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_outputs.last_hidden_state
        
        lstm_output, _ = self.lstm(sequence_output)
        lstm_output = self.dropout(lstm_output)
        emissions = self.hidden2tag(lstm_output)
        
        outputs = {"emissions": emissions}
        
        if labels is not None:
            mask = (labels != -100) & (attention_mask == 1)
            labels_for_crf = labels.clone()
            labels_for_crf[labels == -100] = 0
            loss = -self.crf(emissions, labels_for_crf, mask=mask, reduction='mean')
            outputs["loss"] = loss
        
        with torch.no_grad():
            mask = attention_mask.bool()
            predictions = self.crf.decode(emissions, mask=mask)
            outputs["predictions"] = predictions
            
        return outputs
    
    def get_entities(self, tokens, predictions, word_ids):
        entities = []
        current_entity = None
        word_preds = []
        prev_idx = None
        
        for idx, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx != prev_idx:
                if idx < len(predictions):
                    word_preds.append((word_idx, predictions[idx]))
            prev_idx = word_idx
        
        for word_idx, pred_id in word_preds:
            if word_idx >= len(tokens):
                continue
            label = ID2LABEL.get(pred_id, "O")
            token = tokens[word_idx]
            
            if label.startswith("B-"):
                if current_entity:
                    entities.append(current_entity)
                current_entity = {"text": token, "label": label[2:], "start": word_idx, "end": word_idx}
            elif label.startswith("I-") and current_entity and label[2:] == current_entity["label"]:
                current_entity["text"] += " " + token
                current_entity["end"] = word_idx
            else:
                if current_entity:
                    entities.append(current_entity)
                    current_entity = None
        
        if current_entity:
            entities.append(current_entity)
        return entities

# Test model
model = BertBiLSTMCRF(model_config)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Dataset Preparation

In [ ]:
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast
from sklearn.model_selection import train_test_split
import random

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

class ResumeNERDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        tokens, labels = self.examples[idx]
        
        encoding = self.tokenizer(
            tokens, is_split_into_words=True,
            max_length=self.max_length, padding="max_length",
            truncation=True, return_tensors="pt"
        )
        
        word_ids = encoding.word_ids()
        aligned_labels = []
        prev_word_idx = None
        
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)
            elif word_idx != prev_word_idx:
                if word_idx < len(labels):
                    aligned_labels.append(LABEL2ID[labels[word_idx]])
                else:
                    aligned_labels.append(LABEL2ID["O"])
            else:
                if word_idx < len(labels):
                    label = labels[word_idx]
                    if label.startswith("B-"):
                        label = "I-" + label[2:]
                    aligned_labels.append(LABEL2ID.get(label, LABEL2ID["O"]))
                else:
                    aligned_labels.append(LABEL2ID["O"])
            prev_word_idx = word_idx
        
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long)
        }

In [ ]:
# Generate synthetic training data (if real datasets not available)
def generate_synthetic_data(num_examples=2000):
    skills = ["Python", "JavaScript", "React", "Node.js", "TypeScript", "Java",
              "C++", "SQL", "MongoDB", "PostgreSQL", "AWS", "Docker", "Kubernetes",
              "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "NLP",
              "Git", "Linux", "REST API", "GraphQL", "Redis", "Elasticsearch"]
    companies = ["Google", "Microsoft", "Amazon", "Meta", "Apple", "Netflix",
                 "Uber", "Airbnb", "Stripe", "Spotify"]
    titles = ["Software Engineer", "Senior Developer", "Data Scientist",
              "ML Engineer", "Full Stack Developer", "DevOps Engineer"]
    universities = ["MIT", "Stanford University", "UC Berkeley", "Carnegie Mellon"]
    degrees = ["Bachelor of Science in Computer Science", "Master of Science in Data Science"]
    
    examples = []
    for _ in range(num_examples):
        tokens, labels = [], []
        
        # Name
        tokens.extend(["John", "Doe"])
        labels.extend(["B-NAME", "I-NAME"])
        tokens.append("|")
        labels.append("O")
        
        # Skills
        tokens.extend(["Skills", ":"])
        labels.extend(["O", "O"])
        for i, skill in enumerate(random.sample(skills, random.randint(3, 6))):
            for j, s in enumerate(skill.split()):
                tokens.append(s)
                labels.append("B-SKILL" if j == 0 else "I-SKILL")
            if i < 5:
                tokens.append(",")
                labels.append("O")
        
        # Experience
        tokens.extend(["Experience", ":"])
        labels.extend(["O", "O"])
        title = random.choice(titles)
        for j, t in enumerate(title.split()):
            tokens.append(t)
            labels.append("B-EXP" if j == 0 else "I-EXP")
        tokens.append("at")
        labels.append("O")
        tokens.append(random.choice(companies))
        labels.append("B-ORG")
        
        # Education
        tokens.extend(["Education", ":"])
        labels.extend(["O", "O"])
        degree = random.choice(degrees)
        for j, d in enumerate(degree.split()):
            tokens.append(d)
            labels.append("B-EDU" if j == 0 else "I-EDU")
        tokens.append("from")
        labels.append("O")
        uni = random.choice(universities)
        for j, u in enumerate(uni.split()):
            tokens.append(u)
            labels.append("B-ORG" if j == 0 else "I-ORG")
        
        examples.append((tokens, labels))
    return examples

# Generate data
all_examples = generate_synthetic_data(2000)
print(f"Generated {len(all_examples)} examples")

# Split
train_examples, temp = train_test_split(all_examples, test_size=0.2, random_state=42)
val_examples, test_examples = train_test_split(temp, test_size=0.5, random_state=42)

print(f"Train: {len(train_examples)}, Val: {len(val_examples)}, Test: {len(test_examples)}")

In [ ]:
# Create datasets
train_dataset = ResumeNERDataset(train_examples, tokenizer, train_config.max_seq_length)
val_dataset = ResumeNERDataset(val_examples, tokenizer, train_config.max_seq_length)
test_dataset = ResumeNERDataset(test_examples, tokenizer, train_config.max_seq_length)

train_loader = DataLoader(train_dataset, batch_size=train_config.train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=train_config.eval_batch_size)
test_loader = DataLoader(test_dataset, batch_size=train_config.eval_batch_size)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 6. Training

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm.notebook import tqdm
from seqeval.metrics import f1_score, classification_report
import numpy as np

# Initialize model
model = BertBiLSTMCRF(model_config)
model.to(train_config.device)

# Optimizer with different LR for BERT vs LSTM/CRF
bert_params = [p for n, p in model.named_parameters() if 'bert' in n and p.requires_grad]
other_params = [p for n, p in model.named_parameters() if 'bert' not in n and p.requires_grad]

optimizer = AdamW([
    {'params': bert_params, 'lr': train_config.bert_learning_rate},
    {'params': other_params, 'lr': train_config.lstm_crf_learning_rate}
], weight_decay=train_config.weight_decay)

num_training_steps = len(train_loader) * train_config.num_epochs
num_warmup_steps = int(num_training_steps * train_config.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
)

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            outputs = model(input_ids, attention_mask, labels)
            total_loss += outputs["loss"].item()
            
            for pred_seq, label_seq, mask in zip(
                outputs["predictions"], labels.cpu().numpy(), attention_mask.cpu().numpy()
            ):
                pred_labels, true_labels = [], []
                for pred, label, m in zip(pred_seq, label_seq, mask):
                    if m == 1 and label != -100:
                        pred_labels.append(ID2LABEL[pred])
                        true_labels.append(ID2LABEL[label])
                if pred_labels:
                    all_preds.append(pred_labels)
                    all_labels.append(true_labels)
    
    return {
        "loss": total_loss / len(dataloader),
        "f1": f1_score(all_labels, all_preds),
        "preds": all_preds,
        "labels": all_labels
    }

In [ ]:
# Training loop
best_f1 = 0
patience = 0
history = []

for epoch in range(1, train_config.num_epochs + 1):
    model.train()
    total_loss = 0
    
    progress = tqdm(train_loader, desc=f"Epoch {epoch}")
    for batch in progress:
        input_ids = batch["input_ids"].to(train_config.device)
        attention_mask = batch["attention_mask"].to(train_config.device)
        labels = batch["labels"].to(train_config.device)
        
        outputs = model(input_ids, attention_mask, labels)
        loss = outputs["loss"]
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), train_config.max_grad_norm)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress.set_postfix({"loss": f"{loss.item():.4f}"})
    
    # Evaluate
    val_results = evaluate(model, val_loader, train_config.device)
    
    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch}: Train Loss={avg_loss:.4f}, Val Loss={val_results['loss']:.4f}, Val F1={val_results['f1']:.4f}")
    
    history.append({"epoch": epoch, "train_loss": avg_loss, "val_loss": val_results["loss"], "val_f1": val_results["f1"]})
    
    # Save best model
    if val_results["f1"] > best_f1:
        best_f1 = val_results["f1"]
        patience = 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "config": model_config,
            "label2id": LABEL2ID,
            "id2label": ID2LABEL,
            "f1": best_f1
        }, "best_model.pt")
        print(f"  -> Saved best model (F1: {best_f1:.4f})")
    else:
        patience += 1
    
    if patience >= train_config.early_stopping_patience:
        print(f"Early stopping at epoch {epoch}")
        break

print(f"\nBest F1: {best_f1:.4f}")

## 7. Final Evaluation

In [ ]:
# Load best model
checkpoint = torch.load("best_model.pt")
model.load_state_dict(checkpoint["model_state_dict"])

# Evaluate on test set
test_results = evaluate(model, test_loader, train_config.device)

print(f"Test F1: {test_results['f1']:.4f}")
print(f"Test Loss: {test_results['loss']:.4f}")
print("\nClassification Report:")
print(classification_report(test_results["labels"], test_results["preds"]))

## 8. Test Entity Extraction

In [ ]:
def extract_entities(model, tokenizer, text, device="cuda"):
    model.eval()
    tokens = text.split()
    encoding = tokenizer(
        tokens, is_split_into_words=True,
        max_length=512, padding="max_length", truncation=True, return_tensors="pt"
    )
    
    with torch.no_grad():
        input_ids = encoding["input_ids"].to(device)
        attention_mask = encoding["attention_mask"].to(device)
        outputs = model(input_ids, attention_mask)
    
    return model.get_entities(tokens, outputs["predictions"][0], encoding.word_ids())

# Test
sample_resume = """
John Smith Senior Software Engineer at Google with 5 years of experience
Skills Python JavaScript React TensorFlow AWS Docker Kubernetes
Education Master of Science in Computer Science from Stanford University 2018
Certifications AWS Certified Solutions Architect
"""

entities = extract_entities(model, tokenizer, sample_resume, train_config.device)
print("Extracted Entities:")
for e in entities:
    print(f"  {e['label']}: {e['text']}")

## 9. Export Model

In [ ]:
import json

# Save final model
torch.save({
    "model_state_dict": model.state_dict(),
    "config": model_config,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL
}, "resume_ner_model.pt")

# Save config
with open("model_config.json", "w") as f:
    json.dump({
        "bert_model_name": model_config.bert_model_name,
        "lstm_hidden_size": model_config.lstm_hidden_size,
        "lstm_num_layers": model_config.lstm_num_layers,
        "lstm_dropout": model_config.lstm_dropout,
        "lstm_bidirectional": model_config.lstm_bidirectional,
        "num_labels": NUM_LABELS,
        "label2id": LABEL2ID,
        "id2label": {str(k): v for k, v in ID2LABEL.items()}
    }, f, indent=2)

# Save tokenizer
tokenizer.save_pretrained("tokenizer")

print("Model exported!")
print(f"Model size: {os.path.getsize('resume_ner_model.pt') / 1e6:.1f} MB")

In [ ]:
# Copy to Google Drive
!mkdir -p /content/drive/MyDrive/HireSense
!cp resume_ner_model.pt /content/drive/MyDrive/HireSense/
!cp model_config.json /content/drive/MyDrive/HireSense/
!cp -r tokenizer /content/drive/MyDrive/HireSense/

print("Files saved to Google Drive!")

## 10. Training History

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_loss = [h["val_loss"] for h in history]
val_f1 = [h["val_f1"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, train_loss, label="Train")
ax1.plot(epochs, val_loss, label="Val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.set_title("Loss")

ax2.plot(epochs, val_f1)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1")
ax2.set_title("Validation F1")

plt.tight_layout()
plt.savefig("training_history.png")
plt.show()